# Notebook 05 — Real Benchmark Ingestion

**Repo:** `int_serialization_benchmark-rml`  
**Layer:** `rml_extension/notebooks/`

Notebooks 01–04 built structural and execution-path proxies.

Notebook 05 adds the next layer:

- ingest real benchmark CSV/JSON outputs when available,
- normalize benchmark columns,
- merge observed throughput/latency with RML structural predictions,
- compare predicted regimes against measured behavior,
- export figures and a lab report.

Constraint view:
> useful models become stronger when they meet measured systems behavior.

## Goals

1. Look for benchmark outputs in `rml_extension/results/raw_benchmarks/`.
2. Accept common CSV/JSON shapes from upstream or hand-exported benchmark runs.
3. Normalize columns:
   - distribution
   - implementation
   - operation
   - throughput
   - latency
   - hardware
   - simd mode
4. Merge with Notebook 04 phase metrics.
5. Produce figures:
   - observed throughput by distribution
   - observed latency by distribution
   - predicted coherence vs observed throughput
   - prediction residuals / mismatch map
6. Export CSV, JSON, Markdown report, and PNG outputs.

If no real benchmark file exists yet, the notebook creates a transparent synthetic benchmark table so the pipeline can run end-to-end.

In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd()
candidates = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
    Path("/content/int_serialization_benchmark-rml"),
    Path("/content"),
]

REPO_ROOT = None
for c in candidates:
    if (c / "rml_extension").exists() or (c / "configs").exists():
        REPO_ROOT = c
        break

if REPO_ROOT is None:
    REPO_ROOT = cwd

RML_ROOT = REPO_ROOT / "rml_extension" if (REPO_ROOT / "rml_extension").exists() else REPO_ROOT

RESULTS_DIR = RML_ROOT / "results"
RAW_BENCH_DIR = RESULTS_DIR / "raw_benchmarks"
FIGURES_DIR = RML_ROOT / "figures"
REPORTS_DIR = RML_ROOT / "reports"

for d in [RESULTS_DIR, RAW_BENCH_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("RML_ROOT:", RML_ROOT)
print("RAW_BENCH_DIR:", RAW_BENCH_DIR)

## Load Notebook 04 phase metrics

These provide the RML structural predictions to compare against real benchmark behavior.

In [ ]:
phase_path = RESULTS_DIR / "notebook04_constraint_phase_metrics.csv"

if phase_path.exists():
    phase = pd.read_csv(phase_path)
    print("Loaded:", phase_path)
else:
    print("Notebook 04 phase metrics not found; using fallback phase table.")
    phase = pd.DataFrame([
        {"name": "low_entropy_repeating", "regime": "coherent-local", "coherence_score": 0.75, "fragmentation_score": 0.02, "simd_suitability": 0.26, "scalar_suitability": 0.98},
        {"name": "sequential_ids", "regime": "scalar-favorable", "coherence_score": 0.52, "fragmentation_score": 0.36, "simd_suitability": 0.39, "scalar_suitability": 0.62},
        {"name": "zipfian_smallints", "regime": "simd-favorable", "coherence_score": 0.42, "fragmentation_score": 0.79, "simd_suitability": 0.67, "scalar_suitability": 0.22},
        {"name": "uniform_32bit", "regime": "simd-favorable", "coherence_score": 0.24, "fragmentation_score": 0.95, "simd_suitability": 0.62, "scalar_suitability": 0.08},
        {"name": "clustered_ranges", "regime": "fragmented-irregular", "coherence_score": 0.22, "fragmentation_score": 1.00, "simd_suitability": 0.45, "scalar_suitability": 0.05},
    ])

phase.head()

## Ingest raw benchmark outputs

Place benchmark CSV/JSON files here:

```text
rml_extension/results/raw_benchmarks/
```

Supported flexible column names include:

- distribution / input_distribution / dataset / name
- implementation / algorithm / method
- operation / op
- throughput_mib_s / throughput / mib_s / gb_s
- latency_ns / latency / ns_per_op
- hardware_profile / hardware / platform
- simd / simd_mode / vector_mode

In [ ]:
def read_benchmark_file(path):
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() == ".json":
        try:
            return pd.read_json(path)
        except ValueError:
            data = json.loads(path.read_text())
            return pd.DataFrame(data)
    return None

raw_frames = []
for path in sorted(RAW_BENCH_DIR.glob("*")):
    if path.suffix.lower() not in [".csv", ".json"]:
        continue
    try:
        df_raw = read_benchmark_file(path)
        if df_raw is not None and len(df_raw):
            df_raw["source_file"] = path.name
            raw_frames.append(df_raw)
            print("Loaded raw benchmark:", path.name, df_raw.shape)
    except Exception as e:
        print("Skipping", path.name, "because", e)

if raw_frames:
    raw = pd.concat(raw_frames, ignore_index=True)
else:
    print("No raw benchmark files found; creating synthetic observed benchmark table.")
    raw = pd.DataFrame([
        {"distribution": "low_entropy_repeating", "implementation": "scalar_fastpath", "operation": "serialize", "throughput_mib_s": 1650, "latency_ns": 0.60, "hardware_profile": "baseline", "simd_mode": "scalar"},
        {"distribution": "sequential_ids", "implementation": "scalar_fastpath", "operation": "serialize", "throughput_mib_s": 1350, "latency_ns": 0.74, "hardware_profile": "baseline", "simd_mode": "scalar"},
        {"distribution": "uniform_32bit", "implementation": "simd_path", "operation": "serialize", "throughput_mib_s": 1900, "latency_ns": 0.53, "hardware_profile": "baseline", "simd_mode": "SIMD"},
        {"distribution": "zipfian_smallints", "implementation": "simd_path", "operation": "serialize", "throughput_mib_s": 1500, "latency_ns": 0.67, "hardware_profile": "baseline", "simd_mode": "SIMD"},
        {"distribution": "clustered_ranges", "implementation": "mixed_path", "operation": "serialize", "throughput_mib_s": 950, "latency_ns": 1.05, "hardware_profile": "baseline", "simd_mode": "mixed"},
    ])

raw.head()

## Normalize benchmark columns

In [ ]:
def first_existing(df, names):
    for n in names:
        if n in df.columns:
            return n
    return None

def normalize_benchmark(raw):
    df = raw.copy()
    colmap = {}

    mappings = {
        "distribution": ["distribution", "input_distribution", "dataset", "name"],
        "implementation": ["implementation", "algorithm", "method", "function"],
        "operation": ["operation", "op", "direction"],
        "throughput_mib_s": ["throughput_mib_s", "throughput", "mib_s", "mb_s", "gb_s"],
        "latency_ns": ["latency_ns", "latency", "ns_per_op", "time_ns"],
        "hardware_profile": ["hardware_profile", "hardware", "platform", "machine"],
        "simd_mode": ["simd_mode", "simd", "vector_mode"],
    }

    out = pd.DataFrame()
    for target, candidates in mappings.items():
        src = first_existing(df, candidates)
        if src is not None:
            out[target] = df[src]
        else:
            out[target] = None

    # Convert GB/s to MiB/s if original source was gb_s.
    if "gb_s" in df.columns and "throughput_mib_s" in out.columns:
        out["throughput_mib_s"] = pd.to_numeric(out["throughput_mib_s"], errors="coerce") * 1024

    out["throughput_mib_s"] = pd.to_numeric(out["throughput_mib_s"], errors="coerce")
    out["latency_ns"] = pd.to_numeric(out["latency_ns"], errors="coerce")

    out["distribution"] = out["distribution"].astype(str)
    out["implementation"] = out["implementation"].fillna("unknown").astype(str)
    out["operation"] = out["operation"].fillna("unknown").astype(str)
    out["hardware_profile"] = out["hardware_profile"].fillna("unknown").astype(str)
    out["simd_mode"] = out["simd_mode"].fillna("unknown").astype(str)

    return out.dropna(subset=["distribution"])

bench = normalize_benchmark(raw)
bench

## Aggregate observed benchmark behavior

If multiple runs exist, summarize by distribution.

In [ ]:
agg = (
    bench
    .groupby("distribution", as_index=False)
    .agg(
        observed_throughput_mib_s=("throughput_mib_s", "mean"),
        observed_latency_ns=("latency_ns", "mean"),
        runs=("distribution", "size"),
        implementations=("implementation", lambda x: ", ".join(sorted(set(map(str, x))))),
        simd_modes=("simd_mode", lambda x: ", ".join(sorted(set(map(str, x))))),
        hardware_profiles=("hardware_profile", lambda x: ", ".join(sorted(set(map(str, x))))),
    )
)

agg

## Merge observed benchmarks with RML phase metrics

In [ ]:
merged = agg.merge(phase, left_on="distribution", right_on="name", how="left")

if "coherence_score" not in merged.columns:
    merged["coherence_score"] = np.nan
if "fragmentation_score" not in merged.columns:
    merged["fragmentation_score"] = np.nan
if "simd_suitability" not in merged.columns:
    merged["simd_suitability"] = np.nan
if "scalar_suitability" not in merged.columns:
    merged["scalar_suitability"] = np.nan

# Normalize observed throughput for comparison.
t = merged["observed_throughput_mib_s"].astype(float)
if t.notna().sum() > 1 and t.max() != t.min():
    merged["observed_throughput_norm"] = (t - t.min()) / (t.max() - t.min())
else:
    merged["observed_throughput_norm"] = 0.0

# Mismatch proxy: high predicted coherence but low observed throughput, or vice versa.
merged["coherence_throughput_gap"] = (
    merged["coherence_score"].fillna(0) - merged["observed_throughput_norm"].fillna(0)
)
merged["abs_prediction_gap"] = merged["coherence_throughput_gap"].abs()

merged

## Export merged benchmark table

In [ ]:
csv_path = RESULTS_DIR / "notebook05_real_benchmark_ingestion.csv"
json_path = RESULTS_DIR / "notebook05_real_benchmark_ingestion.json"

merged.to_csv(csv_path, index=False)
merged.to_json(json_path, orient="records", indent=2)

print("Saved:", csv_path)
print("Saved:", json_path)

## Figure 1 — Observed throughput by distribution

In [ ]:
fig_path_1 = FIGURES_DIR / "notebook05_observed_throughput.png"

plot_df = merged.sort_values("observed_throughput_mib_s")
plt.figure(figsize=(9, 5))
plt.bar(plot_df["distribution"], plot_df["observed_throughput_mib_s"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Observed throughput (MiB/s)")
plt.title("Real Benchmark Ingestion: Observed Throughput")
plt.tight_layout()
plt.savefig(fig_path_1, dpi=160)
plt.show()

print("Saved:", fig_path_1)

## Figure 2 — Observed latency by distribution

In [ ]:
fig_path_2 = FIGURES_DIR / "notebook05_observed_latency.png"

plot_df = merged.sort_values("observed_latency_ns")
plt.figure(figsize=(9, 5))
plt.bar(plot_df["distribution"], plot_df["observed_latency_ns"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Observed latency (ns)")
plt.title("Real Benchmark Ingestion: Observed Latency")
plt.tight_layout()
plt.savefig(fig_path_2, dpi=160)
plt.show()

print("Saved:", fig_path_2)

## Figure 3 — Predicted coherence vs observed throughput

In [ ]:
fig_path_3 = FIGURES_DIR / "notebook05_coherence_vs_throughput.png"

plt.figure(figsize=(8, 6))
plt.scatter(merged["coherence_score"], merged["observed_throughput_norm"])
for _, row in merged.iterrows():
    plt.annotate(row["distribution"], (row["coherence_score"], row["observed_throughput_norm"]), fontsize=8)
plt.xlabel("Predicted coherence score")
plt.ylabel("Observed throughput (normalized)")
plt.title("Prediction Check: Coherence vs Observed Throughput")
plt.tight_layout()
plt.savefig(fig_path_3, dpi=160)
plt.show()

print("Saved:", fig_path_3)

## Figure 4 — Prediction gap / mismatch map

In [ ]:
fig_path_4 = FIGURES_DIR / "notebook05_prediction_gap.png"

plot_df = merged.sort_values("abs_prediction_gap")
plt.figure(figsize=(9, 5))
plt.bar(plot_df["distribution"], plot_df["abs_prediction_gap"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("|coherence - normalized throughput|")
plt.title("Prediction Gap: Structural Model vs Observed Throughput")
plt.tight_layout()
plt.savefig(fig_path_4, dpi=160)
plt.show()

print("Saved:", fig_path_4)

## Lab-report summary

In [ ]:
report_path = REPORTS_DIR / "report_05_real_benchmark_ingestion.md"

summary_cols = [
    "distribution", "observed_throughput_mib_s", "observed_latency_ns",
    "runs", "regime", "coherence_score", "fragmentation_score",
    "observed_throughput_norm", "abs_prediction_gap",
    "implementations", "simd_modes", "hardware_profiles"
]
available_summary_cols = [c for c in summary_cols if c in merged.columns]

lines = [
    "# Report 05 — Real Benchmark Ingestion",
    "",
    "This report ingests benchmark outputs and compares observed performance against RML structural phase metrics.",
    "",
    "Constraint view:",
    "> useful models become stronger when they meet measured systems behavior.",
    "",
    "## Generated outputs",
    "",
    f"- Metrics CSV: `{csv_path}`",
    f"- Metrics JSON: `{json_path}`",
    f"- Figure: `{fig_path_1}`",
    f"- Figure: `{fig_path_2}`",
    f"- Figure: `{fig_path_3}`",
    f"- Figure: `{fig_path_4}`",
    "",
    "## Benchmark summary",
    "",
    merged[available_summary_cols].to_markdown(index=False),
    "",
    "## Interpretation",
    "",
    "- Observed throughput and latency are now connected to structural predictions.",
    "- Prediction gaps identify where the proxy model needs correction or real hardware context.",
    "- High coherence does not automatically mean maximum throughput; it means structural alignment under chosen constraints.",
    "- This notebook creates the bridge from RML proxy analysis to real benchmark validation.",
    "",
    "## Next step",
    "",
    "Notebook 06 should add hardware-counter overlays or architecture-specific benchmark comparisons.",
]

report_path.write_text("\n".join(lines))
print("Saved:", report_path)

## Optional: download output bundle in Colab

Uncomment the following cell if you are running this notebook in Google Colab and want to download generated outputs.

In [ ]:
# OPTIONAL COLAB DOWNLOAD
#
# EXPORT_NAME = "notebook05_real_benchmark_ingestion_outputs.zip"
# export_path = RML_ROOT / EXPORT_NAME
#
# with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:
#     for folder in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
#         for p in folder.glob("notebook05_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#         for p in folder.glob("report_05_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#
# from google.colab import files
# files.download(str(export_path))